# Olist E-Commerce Analytics

## Notebook 1: Data Understanding & Initial Exploration

### Objective

The objective of this notebook is to understand the structure of the Olist Brazilian E-commerce dataset before performing any cleaning or analysis.

This includes:

- Understanding each table
- Checking dimensions
- Inspecting data types
- Identifying missing values
- Detecting duplicates
- Understanding relationships between tables

In [1]:
import pandas as pd
import numpy as np

print(pd.__version__)

2.3.3


In [2]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")

orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")

order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")

reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

products = pd.read_csv("../data/raw/olist_products_dataset.csv")

sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")

geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")

category_translation = pd.read_csv(
    "../data/raw/product_category_name_translation.csv"
)

## Dataset Inventory

Before exploring individual tables, it is useful to obtain a high-level overview of all available datasets.

The inventory summarizes the size of each dataset and provides an initial understanding of the overall database structure.

In [5]:
dataset_info = {
    "Customers": {
        "DataFrame": customers,
        "Purpose": "Customer information and location"
    },
    "Orders": {
        "DataFrame": orders,
        "Purpose": "Order lifecycle and status"
    },
    "Order Items": {
        "DataFrame": order_items,
        "Purpose": "Products purchased in each order"
    },
    "Payments": {
        "DataFrame": payments,
        "Purpose": "Payment details for each order"
    },
    "Reviews": {
        "DataFrame": reviews,
        "Purpose": "Customer review scores and comments"
    },
    "Products": {
        "DataFrame": products,
        "Purpose": "Product information"
    },
    "Sellers": {
        "DataFrame": sellers,
        "Purpose": "Seller information"
    },
    "Geolocation": {
        "DataFrame": geolocation,
        "Purpose": "ZIP code location reference"
    },
    "Category Translation": {
        "DataFrame": category_translation,
        "Purpose": "Portuguese to English category mapping"
    }
}

summary = pd.DataFrame({
    "Dataset": dataset_info.keys(),
    "Rows": [v["DataFrame"].shape[0] for v in dataset_info.values()],
    "Columns": [v["DataFrame"].shape[1] for v in dataset_info.values()],
    "Purpose": [v["Purpose"] for v in dataset_info.values()]
})

summary

,Dataset,Rows,Columns,Purpose
0,Customers,99441,5,Customer information and location
1,Orders,99441,8,Order lifecycle and status
2,Order Items,112650,7,Products purchased in each order
3,Payments,103886,5,Payment details for each order
4,Reviews,99224,7,Customer review scores and comments
5,Products,32951,9,Product information
6,Sellers,3095,4,Seller information
7,Geolocation,1000163,5,ZIP code location reference
8,Category Translation,71,2,Portuguese to English category mapping


## Dataset Profiling

Before performing data cleaning, each dataset is profiled to understand its structure and quality.

For every dataset, we examine:

- Shape
- Column names
- Data types
- Missing values
- Duplicate rows
- Sample records

This helps identify potential data quality issues before analysis.

In [6]:
def explore_table(df, table_name):
    """
    Display a summary of a dataset.
    """

    print("=" * 70)
    print(f"{table_name.upper()}")
    print("=" * 70)

    print(f"\nShape: {df.shape}")

    print("\nColumn Names:")
    print(df.columns.tolist())

    print("\nData Types:")
    display(df.dtypes.to_frame(name="Data Type"))

    print("\nMissing Values:")
    display(df.isnull().sum().to_frame(name="Missing Values"))

    print(f"\nDuplicate Rows: {df.duplicated().sum()}")

    print("\nFirst Five Records:")
    display(df.head())

In [7]:
explore_table(orders, "Orders")

ORDERS

Shape: (99441, 8)

Column Names:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Data Types:


,Data Type
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,object
order_approved_at,object
order_delivered_carrier_date,object
order_delivered_customer_date,object
order_estimated_delivery_date,object



Missing Values:


,Missing Values
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0



Duplicate Rows: 0

First Five Records:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


### Orders Table – Initial Observations

- The Orders table contains **99,441 records**, where each row represents one customer order.
- The table appears to be the central transactional (fact) table of the database.
- It includes identifiers (`order_id`, `customer_id`), order status, and multiple timestamps describing the order lifecycle.
- No duplicate rows were found.
- Missing values are present only in delivery-related timestamps, which are likely associated with undelivered or cancelled orders rather than data quality issues.
- All timestamp columns are currently stored as `object` and will require conversion to `datetime` during the data cleaning phase.

In [8]:
orders["order_id"].nunique()

99441

In [9]:
orders.shape[0]

99441

In [10]:
explore_table(customers, "Customers")

CUSTOMERS

Shape: (99441, 5)

Column Names:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Data Types:


,Data Type
customer_id,object
customer_unique_id,object
customer_zip_code_prefix,int64
customer_city,object
customer_state,object



Missing Values:


,Missing Values
customer_id,0
customer_unique_id,0
customer_zip_code_prefix,0
customer_city,0
customer_state,0



Duplicate Rows: 0

First Five Records:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [ ]:
customers["customer_id"].nunique()

99441

In [12]:
customers.shape[0]

99441

In [13]:
customers["customer_unique_id"].nunique()

96096

### Customers Table – Initial Observations

- The Customers table contains **99,441 customer records**.
- Each row represents a **customer record associated with an order**, not necessarily a unique customer.
- `customer_id` is the **Primary Key** of the table.
- `customer_unique_id` identifies the same customer across multiple purchases and customer records.
- Customer location information (city and state) is stored in this table and will later support geographical analysis.
- No duplicate customer records were found based on `customer_id`.

In [14]:
explore_table(order_items, "Order Items")

ORDER ITEMS

Shape: (112650, 7)

Column Names:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Data Types:


,Data Type
order_id,object
order_item_id,int64
product_id,object
seller_id,object
shipping_limit_date,object
price,float64
freight_value,float64



Missing Values:


,Missing Values
order_id,0
order_item_id,0
product_id,0
seller_id,0
shipping_limit_date,0
price,0
freight_value,0



Duplicate Rows: 0

First Five Records:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [15]:
order_items["order_id"].nunique()

98666

In [16]:
order_items["product_id"].nunique()

32951

In [17]:
order_items.shape[0]

112650

In [18]:
explore_table(order_items, "Order Items")

ORDER ITEMS

Shape: (112650, 7)

Column Names:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Data Types:


,Data Type
order_id,object
order_item_id,int64
product_id,object
seller_id,object
shipping_limit_date,object
price,float64
freight_value,float64



Missing Values:


,Missing Values
order_id,0
order_item_id,0
product_id,0
seller_id,0
shipping_limit_date,0
price,0
freight_value,0



Duplicate Rows: 0

First Five Records:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [19]:
order_items["order_id"].nunique()

98666

In [20]:
order_items["product_id"].nunique()

32951

In [21]:
order_items["seller_id"].nunique()

3095

In [22]:
order_items.shape[0]

112650

In [23]:
order_items[["order_id", "product_id"]].drop_duplicates().shape[0]

102425

In [24]:
order_items[["order_id", "order_item_id"]].drop_duplicates().shape[0]

112650

### Order Items Table – Initial Observations

- The Order Items table contains one row for each product purchased within an order (line-item level).
- An order may contain multiple products, resulting in multiple rows with the same `order_id`.
- `order_item_id` identifies the sequence of each item within an order.
- Product price and freight (shipping) cost are stored at the line-item level because each product may have a different price and shipping charge.
- This table links orders, products, and sellers, making it central to product, revenue, and logistics analysis.

In [25]:
explore_table(products, "Products")

PRODUCTS

Shape: (32951, 9)

Column Names:
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

Data Types:


,Data Type
product_id,object
product_category_name,object
product_name_lenght,float64
product_description_lenght,float64
product_photos_qty,float64
product_weight_g,float64
product_length_cm,float64
product_height_cm,float64
product_width_cm,float64



Missing Values:


,Missing Values
product_id,0
product_category_name,610
product_name_lenght,610
product_description_lenght,610
product_photos_qty,610
product_weight_g,2
product_length_cm,2
product_height_cm,2
product_width_cm,2



Duplicate Rows: 0

First Five Records:


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [26]:
products["product_id"].nunique()

32951

In [27]:
products.shape[0]

32951

### Products Table – Initial Observations

- The Products table contains **32,951 product records**.
- Each row represents one unique product.
- `product_id` is the Primary Key of the table.
- The table stores descriptive information such as product category, dimensions, and weight.
- Product dimensions and weight will be useful for logistics and freight analysis.
- Product categories will support category-wise sales and revenue analysis.
- The table does not contain pricing information because prices are stored at the transaction level in the Order Items table.

In [28]:
explore_table(payments, "Payments")

PAYMENTS

Shape: (103886, 5)

Column Names:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Data Types:


,Data Type
order_id,object
payment_sequential,int64
payment_type,object
payment_installments,int64
payment_value,float64



Missing Values:


,Missing Values
order_id,0
payment_sequential,0
payment_type,0
payment_installments,0
payment_value,0



Duplicate Rows: 0

First Five Records:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [29]:
payments["order_id"].nunique()

99440

In [30]:
payments.shape[0]

103886

In [31]:
explore_table(payments, "Payments")

PAYMENTS

Shape: (103886, 5)

Column Names:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Data Types:


,Data Type
order_id,object
payment_sequential,int64
payment_type,object
payment_installments,int64
payment_value,float64



Missing Values:


,Missing Values
order_id,0
payment_sequential,0
payment_type,0
payment_installments,0
payment_value,0



Duplicate Rows: 0

First Five Records:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [32]:
payments["order_id"].nunique()

99440

In [33]:
payments.shape[0]

103886

In [34]:
payments.groupby("order_id").size().sort_values(ascending=False).head()

order_id
fa65dad1b0e818e3ccc5cb0e39231352    29
ccf804e764ed5650cd8759557269dc13    26
285c2e15bebd4ac83635ccc563dc71f4    22
895ab968e7bb0d5659d16cd74cd1650c    21
fedcd9f7ccdc8cba3a18defedd1a5547    19
dtype: int64

In [35]:
explore_table(reviews, "Reviews")

REVIEWS

Shape: (99224, 7)

Column Names:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

Data Types:


,Data Type
review_id,object
order_id,object
review_score,int64
review_comment_title,object
review_comment_message,object
review_creation_date,object
review_answer_timestamp,object



Missing Values:


,Missing Values
review_id,0
order_id,0
review_score,0
review_comment_title,87656
review_comment_message,58247
review_creation_date,0
review_answer_timestamp,0



Duplicate Rows: 0

First Five Records:


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [36]:
reviews["review_id"].nunique()

98410

In [37]:
reviews["order_id"].nunique()

98673

In [38]:
reviews.shape[0]

99224

In [39]:
explore_table(sellers, "Sellers")

SELLERS

Shape: (3095, 4)

Column Names:
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

Data Types:


,Data Type
seller_id,object
seller_zip_code_prefix,int64
seller_city,object
seller_state,object



Missing Values:


,Missing Values
seller_id,0
seller_zip_code_prefix,0
seller_city,0
seller_state,0



Duplicate Rows: 0

First Five Records:


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [40]:
sellers["seller_id"].nunique()

sellers.shape[0]

3095

In [41]:
explore_table(category_translation, "Category Translation")

CATEGORY TRANSLATION

Shape: (71, 2)

Column Names:
['product_category_name', 'product_category_name_english']

Data Types:


,Data Type
product_category_name,object
product_category_name_english,object



Missing Values:


,Missing Values
product_category_name,0
product_category_name_english,0



Duplicate Rows: 0

First Five Records:


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [42]:
explore_table(geolocation, "Geolocation")

GEOLOCATION

Shape: (1000163, 5)

Column Names:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Data Types:


,Data Type
geolocation_zip_code_prefix,int64
geolocation_lat,float64
geolocation_lng,float64
geolocation_city,object
geolocation_state,object



Missing Values:


,Missing Values
geolocation_zip_code_prefix,0
geolocation_lat,0
geolocation_lng,0
geolocation_city,0
geolocation_state,0



Duplicate Rows: 261831

First Five Records:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


# Data Dictionary

In [43]:
data_dictionary = pd.DataFrame({
    "Table": [
        "Customers",
        "Orders",
        "Order Items",
        "Products",
        "Payments",
        "Reviews",
        "Sellers",
        "Category Translation",
        "Geolocation"
    ],

    "Granularity": [
        "One customer record",
        "One order",
        "One product within an order",
        "One product",
        "One payment transaction",
        "One review",
        "One seller",
        "One category mapping",
        "One ZIP code record"
    ],

    "Primary Key": [
        "customer_id",
        "order_id",
        "(order_id, order_item_id)",
        "product_id",
        "(order_id, payment_sequential)",
        "review_id",
        "seller_id",
        "product_category_name",
        "No single primary key"
    ],

    "Purpose": [
        "Stores customer information",
        "Stores order lifecycle",
        "Stores product-level transaction details",
        "Stores product attributes",
        "Stores payment details",
        "Stores customer feedback",
        "Stores seller information",
        "Maps Portuguese categories to English",
        "Stores geographical reference information"
    ]
})

data_dictionary

,Table,Granularity,Primary Key,Purpose
0,Customers,One customer record,customer_id,Stores customer information
1,Orders,One order,order_id,Stores order lifecycle
2,Order Items,One product within an order,"(order_id, order_item_id)",Stores product-level transaction details
3,Products,One product,product_id,Stores product attributes
4,Payments,One payment transaction,"(order_id, payment_sequential)",Stores payment details
5,Reviews,One review,review_id,Stores customer feedback
6,Sellers,One seller,seller_id,Stores seller information
7,Category Translation,One category mapping,product_category_name,Maps Portuguese categories to English
8,Geolocation,One ZIP code record,No single primary key,Stores geographical reference information


# Relationship Map

```
Customers
    │
customer_id
    │
    ▼
Orders
    │
    ├──────── Payments
    │
    ├──────── Reviews
    │
    └──────── Order Items
                │
                ├──────── Products
                │
                └──────── Sellers

Products
    │
product_category_name
    │
    ▼
Category Translation

Customers
    │
customer_zip_code_prefix
    │
    ▼
Geolocation

Sellers
    │
seller_zip_code_prefix
    │
    ▼
Geolocation
```

In [ ]:
# Initial Data Quality Assessment

In [44]:
quality_report = pd.DataFrame({

    "Table": [
        "Customers",
        "Orders",
        "Order Items",
        "Products",
        "Payments",
        "Reviews",
        "Sellers",
        "Category Translation",
        "Geolocation"
    ],

    "Missing Values": [
        customers.isnull().sum().sum(),
        orders.isnull().sum().sum(),
        order_items.isnull().sum().sum(),
        products.isnull().sum().sum(),
        payments.isnull().sum().sum(),
        reviews.isnull().sum().sum(),
        sellers.isnull().sum().sum(),
        category_translation.isnull().sum().sum(),
        geolocation.isnull().sum().sum()
    ],

    "Duplicate Rows": [
        customers.duplicated().sum(),
        orders.duplicated().sum(),
        order_items.duplicated().sum(),
        products.duplicated().sum(),
        payments.duplicated().sum(),
        reviews.duplicated().sum(),
        sellers.duplicated().sum(),
        category_translation.duplicated().sum(),
        geolocation.duplicated().sum()
    ]
})

quality_report

,Table,Missing Values,Duplicate Rows
0,Customers,0,0
1,Orders,4908,0
2,Order Items,0,0
3,Products,2448,0
4,Payments,0,0
5,Reviews,145903,0
6,Sellers,0,0
7,Category Translation,0,0
8,Geolocation,0,261831


# Key Findings from Data Exploration

## Database Structure

- The dataset consists of 9 related tables representing different business entities.
- The Orders table acts as the central fact table connecting customers, products, payments, reviews, and sellers.
- Dimension tables provide descriptive information such as customer details, product attributes, seller information, and geographical locations.

## Data Quality

- Most tables contain very few duplicate records.
- Missing values are mainly observed in delivery-related columns, which are likely due to business processes (e.g., cancelled or undelivered orders).
- Timestamp columns are currently stored as object data types and will require conversion during data cleaning.

## Business Insights

- One order may contain multiple products.
- One order may involve multiple payment transactions.
- Products are linked to sellers through the Order Items table.
- Customer feedback is captured separately in the Reviews table.
- The dataset is well suited for customer, product, payment, logistics, and seller performance analysis.